# Notebook 01: Data Collection
### Pulls AI industry news articles via NewsAPI, cleans them,
### and splits them into three cumulative weekly batches.

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Install Dependencies

In [10]:
!pip install newsapi-python pandas -q

## Imports

In [11]:
import os
import re
import json
import difflib
from datetime import datetime

import pandas as pd
from newsapi import NewsApiClient
from google.colab import userdata

## Config

In [12]:
API_KEY = userdata.get("NEWSAPI_KEY")
newsapi = NewsApiClient(api_key=API_KEY)

DATA_DIR = "/content/drive/MyDrive/thesis-rag-cag/data"
os.makedirs(DATA_DIR, exist_ok=True)

# NewsAPI free tier: /v2/everything, up to 100 results per request,
# max 100 requests/day, articles from roughly the last month.
# Each query is ANDed by default (NewsAPI syntax), with an explicit
# AI anchor plus a NOT clause to push out recurring noise categories
# (movie/TV reviews, academic papers using "artificial" in a non-AI
# sense, weather/insurance "model" releases).
KEYWORDS = [
    "OpenAI AND (\"GPT model\" OR chatbot OR AI) NOT (review OR movie OR film)",
    "Anthropic AND (Claude OR AI) NOT (review OR movie OR film)",
    "\"Google Gemini\" AND AI",
    "(Qwen OR Alibaba) AND AI NOT (review OR movie OR film)",
    "\"AI startup\" AND (funding OR raises OR valuation)",
    "\"artificial intelligence\" AND (startup OR company OR model OR chatbot OR agent) "
    "NOT (genome OR genomics OR storm OR weather OR insurance OR review OR movie OR film)",
]

# Domains that are never real news, even when keyword-matched: package
# registries, code-hosting "Show HN"-style posts, parody/satire blogs,
# and link-aggregator digests.
EXCLUDE_DOMAINS = "pypi.org,github.com,github.io,substack.com,majorgeeks.com,daemonology.net"

# Same list, matched against NewsAPI's `source.name` field as a backup
# in case a mirror/aggregator re-posts the same content under a
# different domain than the one excluded above.
SOURCE_BLACKLIST = {
    "pypi.org", "blogger.com", "github.com", "github.io",
    "substack.com", "majorgeeks.com", "daemonology.net",
}

# Title substrings that mark known noise regardless of which query
# or source produced the article.
OFFTOPIC_TITLE_PATTERNS = [
    "review", "movie", "film", "biopic", "box office",
    "genome", "genomic", "convective storm", "catastrophe risk",
    "crypto", "token surge", "price rally",
    "added to pypi",   # safety net if a mirror re-posts PyPI listings
    "show hn",         # Hacker News "Show HN" posts are project showcases, not news
]

# At least one of these must appear somewhere in the title or body
# text, or the article is dropped as off-topic (catches aggregator
# pages that bundle in an unrelated headline).
AI_SIGNAL_TERMS = [
    r"\bai\b", "artificial intelligence", "chatgpt", "gpt-", "gpt model",
    "llm", "openai", "anthropic", "claude", "gemini", "chatbot",
    "machine learning", "ai agent", "ai model", "ai startup",
]
AI_SIGNAL_RE = re.compile("|".join(AI_SIGNAL_TERMS), re.IGNORECASE)

# Minimum usable text length, and the near-duplicate similarity
# threshold used during dedup.
MIN_TEXT_LENGTH = 200
NEAR_DUP_THRESHOLD = 0.85

# Fraction of non-ASCII characters above which an article is treated
# as non-English and dropped (NewsAPI's language="en" filter isn't
# airtight — a few foreign-language press releases slip through).
NON_ASCII_THRESHOLD = 0.15

## Pull Articles

In [13]:
all_articles = []

for query in KEYWORDS:
    response = newsapi.get_everything(
        q=query,
        language="en",
        sort_by="publishedAt",
        page_size=100,
        exclude_domains=EXCLUDE_DOMAINS,
    )
    print(f"Query: {query!r} -> {response['totalResults']} total, {len(response['articles'])} fetched")
    all_articles.extend(response["articles"])

print(f"\nTotal articles pulled before cleaning: {len(all_articles)}")


Query: 'OpenAI AND ("GPT model" OR chatbot OR AI) NOT (review OR movie OR film)' -> 6947 total, 100 fetched
Query: 'Anthropic AND (Claude OR AI) NOT (review OR movie OR film)' -> 6510 total, 99 fetched
Query: '"Google Gemini" AND AI' -> 347 total, 98 fetched
Query: '(Qwen OR Alibaba) AND AI NOT (review OR movie OR film)' -> 493 total, 100 fetched
Query: '"AI startup" AND (funding OR raises OR valuation)' -> 194 total, 100 fetched
Query: '"artificial intelligence" AND (startup OR company OR model OR chatbot OR agent) NOT (genome OR genomics OR storm OR weather OR insurance OR review OR movie OR film)' -> 6208 total, 99 fetched

Total articles pulled before cleaning: 596


## Clean and Deduplicate

In [14]:
def clean_article(a):
    title = (a.get("title") or "").strip()
    desc = (a.get("description") or "").strip()
    content = (a.get("content") or "").strip()
    # NewsAPI truncates `content` on the free tier; combine description + content
    # for the longest usable text we can get.
    full_text = f"{desc}\n\n{content}".strip()
    return {
        "title": title,
        "text": full_text,
        "source": (a.get("source") or {}).get("name", ""),
        "url": a.get("url", ""),
        "published_at": a.get("publishedAt", ""),
    }


def is_too_short_or_incomplete(article):
    return len(article["text"]) <= MIN_TEXT_LENGTH or not article["title"] or not article["published_at"]


def is_blacklisted_source(article):
    return article["source"].strip().lower() in SOURCE_BLACKLIST


def is_offtopic_title(article):
    title_lower = article["title"].lower()
    return any(pattern in title_lower for pattern in OFFTOPIC_TITLE_PATTERNS)


def has_ai_signal(article):
    combined = f"{article['title']} {article['text']}".lower()
    return bool(AI_SIGNAL_RE.search(combined))


def is_non_english(article):
    text = article["title"] + article["text"]
    if not text:
        return False
    non_ascii = sum(1 for ch in text if ord(ch) > 127)
    return (non_ascii / len(text)) > NON_ASCII_THRESHOLD


def normalize_title(t):
    t = t.lower().strip()
    t = t.split(" | ")[0]  # strip outlet suffixes like " | TechCrunch"
    return t


def dedup_articles(articles):
    # Pass 1: exact-title duplicates (after normalization)
    seen_titles = set()
    exact_deduped = []
    for a in articles:
        key = normalize_title(a["title"])
        if key not in seen_titles:
            seen_titles.add(key)
            exact_deduped.append(a)

    # Pass 2: near-duplicates — the same story syndicated under a
    # slightly different headline, or picked up by two outlets.
    deduped = []
    kept_norm_titles = []
    for a in exact_deduped:
        norm = normalize_title(a["title"])
        is_near_dup = any(
            difflib.SequenceMatcher(None, norm, kept).ratio() >= NEAR_DUP_THRESHOLD
            for kept in kept_norm_titles
        )
        if not is_near_dup:
            deduped.append(a)
            kept_norm_titles.append(norm)

    return exact_deduped, deduped


cleaned = [clean_article(a) for a in all_articles]

filter_steps = [
    ("too short / missing title or date", is_too_short_or_incomplete),
    ("blacklisted source", is_blacklisted_source),
    ("off-topic title pattern", is_offtopic_title),
    ("non-English", is_non_english),
]

for label, predicate in filter_steps:
    before = len(cleaned)
    cleaned = [a for a in cleaned if not predicate(a)]
    print(f"Dropped {before - len(cleaned)} articles ({label})")

before = len(cleaned)
cleaned = [a for a in cleaned if has_ai_signal(a)]
print(f"Dropped {before - len(cleaned)} articles (no AI-related term in title/text)")

exact_deduped, deduped = dedup_articles(cleaned)
print(f"Articles after exact-title dedup: {len(exact_deduped)}")
print(f"Articles after near-duplicate dedup: {len(deduped)}")

Dropped 0 articles (too short / missing title or date)
Dropped 12 articles (blacklisted source)
Dropped 13 articles (off-topic title pattern)
Dropped 1 articles (non-English)
Dropped 77 articles (no AI-related term in title/text)
Articles after exact-title dedup: 378
Articles after near-duplicate dedup: 369


## Sort by date, oldest first

In [15]:
deduped.sort(key=lambda a: a["published_at"])

for i, a in enumerate(deduped):
    a["id"] = f"article_{i:04d}"

raw_path = os.path.join(DATA_DIR, "raw_articles.json")
with open(raw_path, "w") as f:
    json.dump(deduped, f, indent=2)
print(f"\nSaved {len(deduped)} articles to {raw_path}")


Saved 369 articles to /content/drive/MyDrive/thesis-rag-cag/data/raw_articles.json


## Sort, assign IDs, save raw backup

In [16]:
with open("data/raw_articles.json", "w") as f:
  json.dump(deduped, f, indent=2)
print(f"Saved {len(deduped)} articles to data/raw_articles.json")

Saved 369 articles to data/raw_articles.json


## Split into three cumulative weekly batches

In [17]:
n = len(deduped)
print(f"Total usable articles: {n}")

if n < 60:
    print("WARNING: fewer than 60 articles pulled. Consider adding more keyword "
          "queries or widening the date range before proceeding.")

# Target cut points: ~20-30 / ~75 / everything, scaled proportionally
# if volume is lower than expected.
week1_n = min(30, max(15, n // 8))
week2_n = min(75, max(week1_n + 15, n // 3))
week3_n = n  # everything

batches = {
    "week1.json": deduped[:week1_n],
    "week2.json": deduped[:week2_n],
    "week3.json": deduped[:week3_n],
}

for filename, batch in batches.items():
    path = os.path.join(DATA_DIR, filename)
    with open(path, "w") as f:
        json.dump(batch, f, indent=2)
    print(f"{filename}: {len(batch)} articles")

print("\nDone. Next step: Notebook 02, building the question set from data/week3.json.")

Total usable articles: 369
week1.json: 30 articles
week2.json: 75 articles
week3.json: 369 articles

Done. Next step: Notebook 02, building the question set from data/week3.json.
